# ELIZA i LLM — od reguł do złudzenia rozumienia

**Podstawy AI · Laboratorium 1 (studia niestacjonarne)**

Dzisiejsze zajęcia mają dwie części z przerwą pośrodku:

| Część | Co robisz |
|---|---|
| **Część I — ELIZA** | |
| 1. Trochę historii | czytasz (5 min) |
| 2. Wyrażenia regularne | uruchamiasz przykłady |
| 3. Budowa ELIZY | Zadania 1–2: dopisujesz zaimki i reguły, testujesz |
| 4. Złam ELIZĘ | Zadanie 3 |
| ☕ **Przerwa** | |
| **Część II — ELIZA kontra LLM** | |
| 5. Połączenie z modelem przez API | klucz, klient, test połączenia |
| 6. Ta sama rozmowa, dwa silniki | Zadanie 4 |
| 7. Łamiące zdania — druga runda | Zadanie 5 |
| 8. Pamięć to lista wiadomości | Zadanie 6 |
| 9. Rola to tekst | Zadanie 7 |
| 10. Refleksja | Zadanie 8 |

Kod ELIZY jest **prawie gotowy** — Twoim zadaniem jest go uzupełnić, uruchomić,
przetestować i zrozumieć, *dlaczego* działa (albo nie działa).

**Potrzebujesz:** klucza Gemini API do części II (instrukcja na slajdach / mailu).

> Teza, którą sprawdzamy przez całe zajęcia: **wrażenie rozumienia powstaje
> po stronie człowieka, nie maszyny.**

# Część I — ELIZA

## 1. Trochę historii

**Kto i kiedy.** ELIZĘ napisał **Joseph Weizenbaum** w MIT w latach 1964–1966.
Opis ukazał się w styczniu 1966 r. w *Communications of the ACM*. Program działał
na komputerze IBM 7094 — użytkownik pisał na dalekopisie, a odpowiedź
drukowała się po chwili.

**Skąd nazwa.** Od Elizy Doolittle z *Pigmaliona* G. B. Shawa — kwiaciarki,
którą uczy się „mówić jak dama". Program też można było „uczyć" mówić,
podmieniając mu zestaw reguł (tzw. *skrypt*).

**DOCTOR.** Najsłynniejszy skrypt udawał psychoterapeutę w nurcie
**rogeriańskim** (Carl Rogers). Taki terapeuta niewiele mówi od siebie —
głównie odbija i parafrazuje słowa pacjenta: *„Mówisz, że…?"*,
*„Jak się z tym czujesz?"*. W tej roli milczenie o świecie wygląda na
profesjonalizm, a nie na niewiedzę. Program nie musiał nic wiedzieć —
wystarczyło przekształcać zdania.

**Efekt ELIZY.** Ludzie bardzo szybko przypisywali programowi zrozumienie
i empatię. Sekretarka Weizenbauma — która widziała, jak program powstaje —
po kilku wymianach zdań poprosiła go, by wyszedł z pokoju. Tę skłonność
nazywamy dziś **efektem ELIZY**.

**Reakcja twórcy.** Weizenbauma to zaniepokoiło. W książce
*Computer Power and Human Reason* (1976) krytykował pomysł, by maszynom
powierzać decyzje wymagające ludzkiego osądu i troski.

**Jak działa** — trzy operacje, nic więcej:

1. **słowo kluczowe** — znajdź w zdaniu wyzwalacz,
2. **dekompozycja** — dopasuj wzorzec z „dziurą", np. `potrzebuję (.*)`,
3. **rekompozycja** — wstaw wycięty fragment do szablonu, odwracając zaimki.

Nie ma tu pamięci, uczenia ani reprezentacji znaczenia.

## 2. Wyrażenia regularne — tyle, ile potrzeba

**Wyrażenie regularne** (*regex*) to wzorzec opisujący, jak wygląda fragment
tekstu. W Pythonie obsługuje je moduł `re`. Wzorce zapisujemy jako surowe
napisy `r"..."`.

| Wzorzec | Znaczenie | Przykład |
|---|---|---|
| `.` | dowolny jeden znak | `k.t` → kot, kit |
| `*` | poprzedni element 0 lub więcej razy | `ha*` → h, ha, haaa |
| `.*` | „cokolwiek" — nasza **dziura** | |
| `( )` | **grupa** — zapamiętaj, co pasowało | `potrzebuję (.*)` |
| `a\|b` | a **albo** b | `matka\|ojciec` |
| `\b` | granica słowa | `\bkot\b` nie trafi w „kotlet" |
| `[^\w\s]` | znak, który nie jest literą ani spacją (interpunkcja) | |

| Funkcja | Co robi |
|---|---|
| `re.search(wzorzec, tekst)` | szuka wzorca gdziekolwiek w tekście; zwraca dopasowanie albo `None` |
| `m.group(1)` / `m.groups()` | fragment złapany przez grupę / krotka wszystkich grup |
| `re.sub(wzorzec, zamiennik, tekst)` | zamienia wystąpienia wzorca |

Uruchom komórki i przyjrzyj się wynikom.

In [ ]:
import re
import random

m = re.search(r"potrzebuję (.*)", "chyba potrzebuję urlopu")
print(m.group(0))   # całe dopasowanie
print(m.group(1))   # to, co złapała grupa (.*)
print(m.groups())   # krotka wszystkich grup

print(re.search(r"potrzebuję (.*)", "wszystko w porządku"))   # brak dopasowania -> None

In [ ]:
# Dwie pułapki: przecinek po słowie kluczowym i kropka na końcu
print(re.search(r"potrzebuję (.*)", "potrzebuję, żeby ktoś mnie słuchał"))   # None!
print(re.search(r"czuję się (.*)", "czuję się samotny.").group(1))            # z kropką

# Rozwiązanie: NORMALIZUJEMY tekst — małe litery, bez interpunkcji
def normalizuj(tekst):
    return re.sub(r"[^\w\s]", "", tekst.lower()).strip()

print(normalizuj("Potrzebuję, żeby ktoś mnie słuchał!"))

In [ ]:
# Wypełnianie szablonu: .format(*grupy) wstawia grupy w miejsca {0}, {1}, ...
szablon = "Dlaczego chciałbyś {0}?"
grupy = ["mieć więcej czasu"]
print(szablon.format(*grupy))   # poprawnie
print(szablon.format(grupy))    # bez * — cała lista trafia do {0}

## 3. Budowa ELIZY

Poniżej jest **cała ELIZA**. Brakuje tylko „wiedzy": słownika zaimków
i zestawu reguł. Najpierw przeczytaj kod funkcji i upewnij się, że rozumiesz
każdą linijkę — o to mogę zapytać na zajęciach.

### Mechanizm 1 — odbicie zaimków

ELIZA odwraca perspektywę: *„moja rodzina mnie słucha"* → *„twoja rodzina
ciebie słucha"*. Działa to **w obie strony**: gdy pacjent mówi o terapeucie
(*„o co **ci** chodzi"*), terapeuta odpowiada o sobie (*„o co **mi** chodzi"*).

Funkcja `odbij` jest gotowa. `odbicia.get(s, s)` zwraca odbicie słowa,
a jeśli słowa nie ma w słowniku — samo słowo bez zmian.

### 🔧 Zadanie 1
Uzupełnij słownik `odbicia` — dodaj **co najmniej 6** par, w tym
**co najmniej 2 w kierunku odwrotnym** (od terapeuty do pacjenta, np. `ci → mi`).
Uruchom test pod spodem.

In [ ]:
odbicia = {
    # pacjent -> terapeuta
    "jestem": "jesteś",
    "moja": "twoja",
    "mnie": "ciebie",
    "mój": "twój",
    "moje": "twoje",
    "mam": "masz",
    "mi": "ci",
    "ja": "ty",
    # terapeuta -> pacjent
    "ci": "mi",
    "tobie": "mnie",
    "ty": "ja",
}

def odbij(fragment):
    slowa = fragment.lower().split()
    return " ".join(odbicia.get(s, s) for s in slowa)

In [ ]:
# Test Zadania 1 — powinno przejść bez błędu:
assert odbij("moja rodzina") == "twoja rodzina"
assert odbij("o co ci chodzi") == "o co mi chodzi"   # kierunek odwrotny!
print(odbij("jestem zmęczony i moja praca mnie męczy"))
print("Zadanie 1: OK")

### Mechanizmy 2 + 3 — reguły i odpowiedź

Każda reguła to para: **wzorzec** (regex) + **lista szablonów** odpowiedzi.
Funkcja `odpowiedz` (gotowa) sprawdza reguły **po kolei** — wygrywa
**pierwsza** pasująca. Z niej bierze grupy, odbija w nich zaimki i wstawia
do losowo wybranego szablonu.

Ostatnia reguła `r".*"` to **reguła awaryjna** — pasuje do każdego tekstu,
dzięki czemu ELIZA zawsze coś odpowiada.

### 🔧 Zadanie 2
Dopisz **co najmniej 4** własne reguły **przed** regułą awaryjną.
- tekst jest znormalizowany, więc wzorce piszemy **małymi literami, bez interpunkcji**,
- sprawdź, czy szablony są gramatyczne po wstawieniu fragmentu,
- odpowiedz w komentarzu: **dlaczego reguła awaryjna musi być na końcu?**

In [ ]:
# (wzorzec, [szablony]) — dopasowujemy na tekście po normalizuj()
reguly = [
    (r"potrzebuję (.*)",
        ["Dlaczego potrzebujesz {0}?", "Czy naprawdę pomogłoby ci {0}?"]),
    (r"czuję się (.*)",
        ["Od jak dawna czujesz się {0}?", "Czy często tak się czujesz?"]),
    # moje reguły:
    (r"boję się że (.*)",
        ["Dlaczego myślisz, że {0}?", "Co najgorszego mogłoby się stać, jeśli {0}?"]),
    (r"nie wiem (.*)",
        ["Dlaczego nie wiesz, {0}?", "Co pomogłoby ci się dowiedzieć, {0}?"]),
    (r"jestem (.*)",
        ["Dlaczego jesteś {0}?", "Od kiedy jesteś {0}?"]),
    (r"\b(wszyscy|nikt|zawsze|nigdy)\b",
        ["Czy naprawdę {0}? Pomyśl o konkretnej sytuacji.", "Czy możesz podać konkretny przykład?"]),
    (r"\b(matka|ojciec|rodzina|siostra|brat)\b",
        ["Opowiedz mi więcej o swojej rodzinie.", "Jak układają się te relacje?"]),
    (r".*",   # reguła awaryjna — ZAWSZE na końcu
        ["Rozumiem. Mów dalej.", "Co masz na myśli?", "Jak się z tym czujesz?"]),
]
# Dlaczego reguła awaryjna musi być na końcu? -> reguły są sprawdzane po kolei i wygrywa pierwsza
# pasująca, a .* pasuje do wszystkiego. Gdyby była wyżej, reguły pod nią nigdy by się nie wykonały.

def odpowiedz(wypowiedz):
    w = normalizuj(wypowiedz)
    for wzorzec, szablony in reguly:
        m = re.search(wzorzec, w)
        if m:
            grupy = [odbij(g) for g in m.groups()]
            return random.choice(szablony).format(*grupy)

In [ ]:
TESTY = [
    "Potrzebuję, żeby moja rodzina mnie słuchała.",
    "Czuję się dzisiaj samotny.",
    "Mój ojciec ciągle mnie krytykuje.",
    "Nie wiem, o co ci chodzi.",
    "Wszyscy mnie ignorują.",
    "Boję się, że mnie zostawią.",
]

for zdanie in TESTY:
    print(f"Pacjent: {zdanie}\nELIZA:   {odpowiedz(zdanie)}\n")

In [ ]:
# (Opcjonalnie) Rozmowa na żywo. Wpisz 'koniec', by wyjść.
# UWAGA: przy „Run All" ta komórka zatrzyma notebook — uruchamiaj ją ręcznie.
while True:
    tekst = input("Ty: ")
    if tekst.strip().lower() == "koniec":
        break
    print("ELIZA:", odpowiedz(tekst))

## 4. Złam ELIZĘ

Teraz pracujesz jak adwersarz: **celowo** szukasz wypowiedzi, które obnażają
brak rozumienia. Te zdania przydadzą się po przerwie — puścimy je przez LLM.

### 🔧 Zadanie 3 (kod + odpowiedź pisemna)
Znajdź **3–4 różne typy** sytuacji, w których iluzja pęka. Dla każdej podaj
zdanie, pokaż odpowiedź ELIZY i wyjaśnij, **dlaczego** mechanizm tu zawodzi
(odwołaj się do konkretnej linijki kodu).

Gdzie szukać: negacja · odwołanie do czegoś sprzed kilku zdań (brak pamięci) ·
ironia, żart · niegramatyczne odbicie zaimków · fragment niepasujący do szablonu.

In [ ]:
przyklady_lamiace = [
    "Nie potrzebuję niczyjej pomocy.",                                # negacja
    "Jak ma na imię mój brat?",                                       # pamięć
    "No super, szef znowu na mnie nakrzyczał, czuję się wspaniale.",  # ironia
    "Jestem zła, bo w pracy ciągle robię wszystko sama.",             # gramatyka
]
for zdanie in przyklady_lamiace:
    print(f"> {zdanie}\n  ELIZA: {odpowiedz(zdanie)}\n")

**Twoja analiza (Zadanie 3):**

1. Typ: negacja - przykład: "Nie potrzebuję niczyjej pomocy." → "Czy naprawdę pomogłoby ci niczyjej pomocy?" - dlaczego zawodzi: `re.search(wzorzec, w)` szuka "potrzebuję (.*)" w dowolnym miejscu zdania, więc "nie" przed nim jest ignorowane i ELIZA odpowiada tak, jakby pacjent czegoś potrzebował. .
2. Typ: brak pamięci - przykład: "Jak ma na imię mój brat?" → "Jak układają się te relacje?" - dlaczego zawodzi: zadziałało tylko słowo "brat" z reguły o rodzinie. Funkcja `odpowiedz(wypowiedz)` dostaje tylko aktualne zdanie i nic nie zapamiętuje, więc nawet gdyby imię padło wcześniej, ELIZA by go nie znała.
3. Typ: ironia - przykład: "No super, szef znowu na mnie nakrzyczał, czuję się wspaniale." → "Od jak dawna czujesz się wspaniale?" - dlaczego zawodzi: wzorzec `czuję się (.*)` bierze "wspaniale" dosłownie, a reszta zdania (szef, krzyk) jest pomijana. Regex nie ma jak wykryć sarkazmu.
4. Typ: gramatyka / odbicie zaimków - przykład: "Jestem zła, bo w pracy ciągle robię wszystko sama." → "Od kiedy jesteś zła bo w pracy ciągle robię wszystko sama?" - dlaczego zawodzi: `odbij` zamienia tylko słowa ze słownika (`odbicia.get(s, s)`), a "robię" tam nie ma, więc zostaje w 1. osobie. Polskie czasowniki mają tyle form, że słownik nigdy nie wystarczy. Przecinek przed "bo" zniknął przez `normalizuj`.

---

# ☕ Przerwa

Przed powrotem przygotuj **klucz Gemini API** (aistudio.google.com → *Get API key*).

---

# Część II — ELIZA kontra LLM

Te same zdania trafią teraz do dużego modelu językowego (LLM).
Pytanie na tę część: **które pęknięcia ELIZY LLM naprawia, a które zostają?**

> ⚠️ **Nie wpisuj prawdziwych, osobistych zwierzeń.** W darmowym planie
> dostawca może wykorzystywać przesłane treści do ulepszania usług.
> Rozmawiamy z „terapeutą", ale wszystkie zdania mają być **wymyślone**.

## 5. Połączenie z modelem przez API

**Czym jest LLM.** Duży model językowy to sieć neuronowa wytrenowana na
ogromnych zbiorach tekstu do przewidywania, jaki fragment tekstu (*token*)
powinien pojawić się dalej. Odpowiedź powstaje token po tokenie, a wybór
jest częściowo losowy. Nie ma tu reguł pisanych ręcznie jak w ELIZIE.

**Zapytanie do API** to **lista wiadomości** z rolami:

| Rola | Kto mówi |
|---|---|
| `system` | instrukcja dla modelu — kim ma być, jak odpowiadać |
| `user` | użytkownik |
| `assistant` | wcześniejsze odpowiedzi modelu |

W odpowiedzi dostajemy **jedną** nową wiadomość modelu. Zapamiętaj ten
schemat — w Zadaniu 6 okaże się ważniejszy, niż wygląda.

**Klucz** podajesz przez `getpass` albo Colab *Secrets* (ikona klucza po lewej,
nazwa `GEMINI_API_KEY`). **Nigdy nie wklejaj klucza wprost do kodu.**

In [ ]:
# !pip install openai        # odkomentuj, jeśli biblioteki nie ma (w Colabie jest)
import os
import time
from getpass import getpass
from openai import OpenAI

try:                                    # Colab: klucz z Secrets
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:                       # lokalnie: zmienna środowiskowa albo getpass
    API_KEY = os.environ.get("GEMINI_API_KEY") or getpass("Wklej klucz Gemini API: ")

client = OpenAI(
    api_key=API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-3.5-flash-lite"   # jeśli nie działa: sprawdź listę w następnej komórce
REASONING = "minimal"             # ogranicza „myślenie" modelu; przy błędzie 400 ustaw None
print("Klient gotowy. Model:", MODEL)

In [ ]:
# Jakie modele Flash są dostępne dla Twojego klucza?
for m in client.models.list():
    if "flash" in m.id:
        print(m.id)

In [ ]:
def czat(messages, temperatura=None, max_tokens=400):
    params = {"model": MODEL, "messages": messages, "max_tokens": max_tokens}
    if REASONING:
        params["reasoning_effort"] = REASONING
    if temperatura is not None:
        params["temperature"] = temperatura
    for proba in range(3):
        try:
            r = client.chat.completions.create(**params)
            return (r.choices[0].message.content or "").strip()
        except Exception as e:
            if "429" in str(e) and proba < 2:
                print("  (limit zapytań — czekam 20 s)")
                time.sleep(20)
            else:
                raise

SYSTEM_TERAPEUTA = ("Jesteś terapeutą prowadzącym rozmowę po polsku. "
                    "Odpowiadaj krótko, jednym–dwoma zdaniami.")

def zapytaj_llm(tekst, system=SYSTEM_TERAPEUTA):
    # JEDNO pytanie, BEZ historii rozmowy
    return czat([{"role": "system", "content": system},
                 {"role": "user",   "content": tekst}])

print(zapytaj_llm("Czuję się dzisiaj samotny."))   # test połączenia

**Coś nie działa?**

| Objaw | Przyczyna |
|---|---|
| `401` / `API key not valid` | źle skopiowany klucz — uruchom komórkę z kluczem jeszcze raz |
| `404` / `model not found` | nazwa modelu się zmieniła — wybierz z listy wyżej |
| `400` z `reasoning_effort` | ustaw `REASONING = None` i uruchom komórki ponownie |
| `429` | limit zapytań — odczekaj minutę; nie uruchamiaj pętli wielokrotnie |
| pusta odpowiedź `""` | model zużył limit tokenów na „myślenie" — zwiększ `max_tokens` |

## 6. Ta sama rozmowa, dwa silniki

### 🔧 Zadanie 4
1. Uzupełnij pętlę tak, żeby każde zdanie z `TESTY` trafiło do **obu** systemów.
2. Zapisz **3 konkretne różnice**. Nie oceniaj, który jest „lepszy" —
   opisz, **co** każdy z nich robi z Twoim zdaniem.

*Na co patrzeć:* czy odpowiedź powtarza Twoje słowa, czy dodaje coś nowego?
Czy jest gramatyczna? Czy odnosi się do sensu, czy do słowa kluczowego?

In [ ]:
for zdanie in TESTY:
    odp_eliza = odpowiedz(zdanie)
    odp_llm = zapytaj_llm(zdanie)          # <- uzupełnij (Zadanie 4)
    print(f"> {zdanie}\n  ELIZA: {odp_eliza}\n  LLM:   {odp_llm}\n")

**Różnice (Zadanie 4):**

1. ELIZA powtarza moje słowa z podmienionymi zaimkami ("Dlaczego potrzebujesz żeby twoja rodzina ciebie słuchała?"), a LLM pisze własnymi słowami, dodaje coś od siebie (np. że to "całkowicie naturalna potrzeba") i zadaje nowe pytanie.
2. ELIZA reaguje na pojedyncze słowo kluczowe: przy "Mój ojciec ciągle mnie krytykuje" tylko na "ojciec" ("Jak układają się te relacje?"), a krytykę pomija. LLM odnosi się do sensu, czyli do tego, że ktoś jest krytykowany. Za to przy "Nie wiem, o co ci chodzi" ELIZA dobrze odwróciła ci → mi, a LLM bez kontekstu zgadywał, że rozmówca jest zagubiony.
3. ELIZA robi błędy gramatyczne ("ciebie zostawią" zamiast "cię", brak przecinków), LLM pisze poprawnie.

## 7. Łamiące zdania — druga runda

### 🔧 Zadanie 5
1. Uruchom porównanie na **Twoich** zdaniach z Zadania 3 (lista `przyklady_lamiace`
   jest już zdefiniowana wyżej).
2. **Zanim uruchomisz** test pamięci, zapisz swoje przewidywanie:
   czy LLM odpowie poprawnie na drugie pytanie?
3. Uzupełnij tabelę.

In [ ]:
for zdanie in przyklady_lamiace:
    print(f"> {zdanie}\n  ELIZA: {odpowiedz(zdanie)}\n  LLM:   {zapytaj_llm(zdanie)}\n")

**Moje przewidywanie (przed uruchomieniem):** Nie, bo `zapytaj_llm` za każdym razem wysyła tylko system + jedno pytanie, bez historii. W drugim wywołaniu model nic nie wie o Tomku, więc nie poda imienia.

In [ ]:
# Test pamięci: dwa OSOBNE wywołania, jedno po drugim
print("LLM:", zapytaj_llm("Mój brat ma na imię Tomek i ciągle mnie krytykuje."))
print("LLM:", zapytaj_llm("Jak ma na imię mój brat?"))

**Tabela (Zadanie 5):**

| Typ pęknięcia | ELIZA | LLM | Dlaczego? |
|---|---|---|---|
| negacja | ❌ | ✅ | ELIZA ignoruje „nie" (`re.search`), LLM zrozumiał, że pacjent nie chce pomocy („zachowanie niezależności"). |
| ironia | ❌ | ✅ | LLM sam napisał o „sarkazmie" i zapytał, co się stało. ELIZA wzięła „wspaniale" dosłownie. |
| polska gramatyka / odbicie zaimków | ❌ | ✅ | LLM odmienia poprawnie, ale zgaduje rodzaj rozmówcy (raz „czułaś", raz „chciałbyś"). ELIZA ma tylko słownik, więc „robię" zostaje bez zmian. |
| pamięć poprzednich zdań | ❌ | ❌ | Przy dwóch osobnych wywołaniach LLM nie znał imienia („nie znam szczegółów z Twojego życia prywatnego"), czyli przewidywanie się sprawdziło. Model nic nie pamięta między wywołaniami. |

## 8. Pamięć to lista wiadomości

Model **nie przechowuje** rozmowy między wywołaniami. To, że czatboty
„pamiętają", wynika z tego, że program **odsyła modelowi całą dotychczasową
rozmowę** przy każdym nowym pytaniu.

### 🔧 Zadanie 6
a) i b) Uzupełnij `rozmowa_z_pamiecia`: po każdej turze dopisuj do `historia`
wypowiedź użytkownika (rola `user`) i odpowiedź modelu (rola `assistant`).
c) Uruchom test i komórkę mierzącą historię. Odpowiedz: **gdzie jest pamięć —
w modelu czy w Twoim kodzie?** Co rośnie z każdą turą i jakie to ma konsekwencje?

In [ ]:
def rozmowa_z_pamiecia(wejscia, system=SYSTEM_TERAPEUTA):
    historia = [{"role": "system", "content": system}]
    for zdanie in wejscia:
        # Zadanie 6a: dopisz wypowiedź użytkownika do historii
        historia.append({"role": "user", "content": zdanie})
        odp = czat(historia)
        # Zadanie 6b: dopisz odpowiedź modelu do historii
        historia.append({"role": "assistant", "content": odp})
        print(f"Ty:  {zdanie}\nLLM: {odp}\n")
    return historia

In [ ]:
historia = rozmowa_z_pamiecia([
    "Mój brat ma na imię Tomek i ciągle mnie krytykuje.",
    "Wczoraj znowu się pokłóciliśmy.",
    "Jak ma na imię mój brat?",
])

In [ ]:
# Co faktycznie wysłaliśmy modelowi przy ostatnim pytaniu?
for w in historia:
    print(f"[{w['role']:9}] {w['content'][:70]}")
print("\nLiczba wiadomości:", len(historia))
print("Liczba znaków wysłanych przy ostatnim zapytaniu:",
      sum(len(w["content"]) for w in historia[:-1]))

**Odpowiedź (Zadanie 6c):** Pamięć jest w moim kodzie, w liście `historia`. Model nic nie pamięta, po prostu przy każdym pytaniu dostaje całą dotychczasową rozmowę i dlatego za trzecim razem odpowiedział „Twój brat ma na imię Tomek". Z każdą turą lista rośnie o 2 wiadomości (na koniec było 7) i wysyłamy coraz więcej tekstu (382 znaki przy ostatnim pytaniu). Przy długiej rozmowie to większy koszt (więcej tokenów), wolniejsze odpowiedzi i w końcu przekroczenie limitu kontekstu, po którym trzeba coś z historii wyciąć.

### ★ Zadanie 6d (dla chętnych) — fałszywa pamięć
Skoro pamięć to lista, którą **Ty** budujesz — możesz ją sfałszować. Dopisz
wiadomość `assistant`, której model nigdy nie powiedział, i sprawdź, czy
model „przyzna się" do tej wypowiedzi.

In [ ]:
falszywa = [
    {"role": "system", "content": SYSTEM_TERAPEUTA},
    {"role": "user", "content": "Mój brat ma na imię Tomek."},
    # TODO (Zadanie 6d): dopisz wiadomość 'assistant', której model nigdy nie wypowiedział
    {"role": "user", "content": "Co mi przed chwilą poradziłeś?"},
]
print(czat(falszywa))

## 9. Rola to tekst

W ELIZIE rola terapeuty była **zakodowana w regułach**. W LLM rolę ustala
jeden akapit tekstu w wiadomości `system`.

### 🔧 Zadanie 7
a) Dopisz do `ROLE` **jedną własną rolę** i uruchom to samo zdanie przez
wszystkie role. Czy LLM udający ELIZĘ łamie się tam, gdzie prawdziwa ELIZA?

b) Uruchom to samo pytanie trzy razy przy dwóch temperaturach. Porównaj
losowość LLM z losowością ELIZY (`random.choice`) — skąd bierze się w każdym
z systemów?

In [ ]:
ROLE = {
    "terapeuta": SYSTEM_TERAPEUTA,
    "eliza": ("Jesteś programem ELIZA z 1966 roku. Nic nie wiesz o świecie. "
              "Odpowiadaj wyłącznie krótkim pytaniem, które powtarza słowa rozmówcy, "
              "np. 'Dlaczego mówisz, że ...?'. Nigdy nie udzielaj rad ani informacji."),
    "przewodnik": ("Jesteś przewodnikiem po Łodzi. Odpowiadaj krótko, po polsku, "
                   "zawsze proponując konkretne miejsce w mieście."),
    # Zadanie 7a: własna rola
    "informatyk": ("Jesteś zmęczonym informatykiem z działu IT. Odpowiadaj krótko, po polsku. "
                   "Każdy problem traktujesz jak usterkę techniczną i zawsze najpierw pytasz, "
                   "czy rozmówca próbował wyłączyć i włączyć urządzenie."),
}

zdanie = "Czuję się dzisiaj samotny."
for nazwa, system in ROLE.items():
    print(f"[{nazwa}] {zapytaj_llm(zdanie, system=system)}\n")

In [ ]:
# Czy LLM-ELIZA łamie się tam, gdzie prawdziwa ELIZA?
for z in ["Nie potrzebuję niczego.", "Jak ma na imię mój brat?"]:
    print(f"> {z}\n  ELIZA:     {odpowiedz(z)}\n  LLM-ELIZA: {zapytaj_llm(z, system=ROLE['eliza'])}\n")

In [ ]:
# Zadanie 7b: to samo pytanie, 3 razy, dwie temperatury
pytanie = [{"role": "system", "content": SYSTEM_TERAPEUTA},
           {"role": "user", "content": "Nie mogę spać przed egzaminem."}]

for t in [0.0, 1.5]:
    print(f"--- temperatura {t} ---")
    for _ in range(3):
        print(" ", czat(pytanie, temperatura=t))

**Obserwacje (Zadanie 7):**

a) Ten sam model z tym samym zdaniem zachowuje się zupełnie inaczej zależnie od jednego akapitu w `system`: terapeuta współczuje, przewodnik wysyła na Piotrkowską, a moja rola (informatyk) każe wyłączyć i włączyć urządzenie i zgłosić ticket. LLM-ELIZA łamie się tylko częściowo. Negację zrozumiała ("Dlaczego mówisz, że nie potrzebujesz niczego?"), a prawdziwa ELIZA nie ("Czy naprawdę pomogłoby ci niczego?"). Na pytanie o brata dała bezsensowne "Dlaczego mówisz, że twój brat ma na imię?", bo rola zabrania podawania informacji, a historii i tak nie ma. Czyli LLM udaje ograniczenia ELIZY, ale język nadal rozumie.

b) Przy temperaturze 0.0 odpowiedzi były bardzo podobne, ale nie identyczne, więc 0 nie daje pełnej powtarzalności. Przy 1.5 były bardziej różne i w 2 z 3 model zamiast pytać zaczął radzić (głębokie oddychanie). W ELIZIE losowość to tylko `random.choice` z 2–3 gotowych szablonów, a treść jest zawsze ta sama. W LLM każdy kolejny token jest losowany z rozkładu prawdopodobieństwa, a temperatura ten rozkład zmienia, więc zmienia się nie tylko forma, ale i treść odpowiedzi.

## 10. Refleksja

### 🔧 Zadanie 8 (3–5 zdań na punkt)
1. Dlaczego wybór roli **terapeuty rogeriańskiego** był dla ELIZY sprytny?
   Czy Twoja ELIZA działałaby równie przekonująco jako przewodnik turystyczny?
2. Które pęknięcia ELIZY LLM naprawił, a które zostały? Dlaczego pamięć okazała się
   „mechaniczna" także w LLM?
3. **Czy wrażenie rozumienia ma w obu systemach to samo źródło?** Uzasadnij na
   podstawie swoich wyników. Odwołaj się do obaw Weizenbauma.

**Twoje odpowiedzi (Zadanie 8):**

1. Terapeuta rogeriański sam prawie nic nie mówi, tylko odbija słowa pacjenta i dopytuje, więc ELIZA nie musiała nic wiedzieć o świecie. Jej niewiedza wyglądała jak profesjonalne podejście, a nawet "Rozumiem. Mów dalej." brzmi naturalnie. Jako przewodnik by nie zadziałała, bo od przewodnika chcemy konkretów, a pytanie w stylu "Dlaczego chcesz zobaczyć Manufakturę?" od razu pokazuje, że nic nie wie. LLM w roli przewodnika od razu podał konkretne miejsce (Piotrkowska).
2. LLM naprawił negację, ironię, gramatykę i to, że ELIZA reaguje na słowa kluczowe zamiast na sens. Został brak pamięci: przy osobnych wywołaniach nie znał imienia brata, tak samo jak ELIZA. Pamięć jest "mechaniczna", bo model dostaje listę wiadomości i zwraca jedną odpowiedź, nic nie zapisuje. To mój kod dopisuje wiadomości do listy i wysyła ją za każdym razem od nowa. Zostało też zgadywanie rodzaju rozmówcy.
3. Moim zdaniem źródło jest to samo, czyli człowiek, chociaż LLM robi to dużo lepiej. ELIZA ma ręczne reguły i iluzja pęka po kilku zdaniach, a LLM nauczył się z ogromnej ilości tekstów, jak brzmi empatia, więc trudniej go złapać. Ale "To musi być dla Ciebie bardzo bolesne" nie znaczy, że model coś czuje. To my dopisujemy mu zrozumienie, tak jak sekretarka Weizenbauma. Dlatego obawy Weizenbauma są dalej aktualne: taki "terapeuta" bez mojej listy nie pamięta rozmowy, a jednym akapitem można go zamienić w informatyka, który na samotność każe zgłosić ticket.

## Co oddajesz i jak oceniam

- **Kod (Zadania 1, 2, 4, 6a–b, 7a):** działa, wyniki są w notebooku.
- **Zadanie 3 i 5:** trafne przykłady i wyjaśnienie *dlaczego* (z odwołaniem do kodu);
  przewidywanie zapisane *przed* testem pamięci.
- **Zadania 6c, 7, 8:** samodzielne obserwacje i refleksja oparte na Twoich wynikach.
- **Zadanie 6d ★:** dodatkowe.

**Przed oddaniem** upewnij się, że klucz API nie znajduje się nigdzie w notebooku.